<a href="https://colab.research.google.com/github/victorm0202/temas_selectos_CD_2026/blob/main/Notebooks/1-7.%20LSTM%20language%20model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelo de lenguaje con LSTM

En éste ejemplo, usaremos una red LSTM con WikiText-2 para predecir el siguiente token a partir de los anteriores. Usaremos éste modelo para generar texto a partir de una frase inicial.

El pipeline incluye preparar el vocabulario, organizar las secuencias, entrenar el modelo y evaluar su perplejidad.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
from tqdm import tqdm
import datasets

## WikiText-2

Cargamos la versión `wikitext-2-raw-v1`, con particiones de entrenamiento, validación y prueba.

In [ ]:
dataset = datasets.load_dataset('wikitext', 'wikitext-2-raw-v1')
print(dataset)
print(dataset['train'][88]['text'])

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})
 This ammunition , and that which I brought with me , was rapidly prepared for use at the Laboratory established at the Little Rock Arsenal for that purpose . As illustrating as the pitiful scarcity of material in the country , the fact may be stated that it was found necessary to use public documents of the State Library for cartridge paper . Gunsmiths were employed or conscripted , tools purchased or impressed , and the repair of the damaged guns I brought with me and about an equal number found at Little Rock commenced at once . But , after inspecting the work and observing the spirit of the men I decided that a garrison 500 strong could hold out against Fitch and that I would lead the remainder - about 1500 - to Gen 'l Rust as 

## Tokenizador y vocabulario

Entrenamos un tokenizador `WordLevel` solo con los textos no vacíos de entrenamiento. Convertimos a minúsculas y separamos palabras y signos de puntuación con `Whitespace`.

Conservamos los tokens que aparecen al menos tres veces. Reservamos `<unk>` para tokens desconocidos y `<eos>` para marcar el final de cada texto.

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.normalizers import Lowercase
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import WordLevelTrainer

tokenizer = Tokenizer(WordLevel(unk_token="<unk>"))
tokenizer.normalizer = Lowercase()
tokenizer.pre_tokenizer = Whitespace()

trainer = WordLevelTrainer(
    min_frequency=3,
    special_tokens=["<unk>", "<eos>"]
)

texts = (example["text"] for example in dataset["train"] if example["text"].strip())
tokenizer.train_from_iterator(texts, trainer=trainer)

unk_id = tokenizer.token_to_id("<unk>")
eos_id = tokenizer.token_to_id("<eos>")
vocab_size = tokenizer.get_vocab_size()

print(vocab_size)
print([tokenizer.id_to_token(i) for i in range(10)])

29476
['<unk>', '<eos>', 'the', ',', '.', 'of', 'and', 'in', 'to', 'a']


## Codificación de los textos

Convertimos cada texto en índices del vocabulario y añadimos `<eos>` al final. Los textos vacíos producen listas vacías. Aplicamos la transformación a las tres particiones y comprobamos un ejemplo mostrando sus índices y tokens.

In [ ]:
def encode_batch(batch):
    return {
        "ids": [
            tokenizer.encode(text).ids + [eos_id] if text.strip() else []
            for text in batch["text"]
        ]
    }

encoded_dataset = dataset.map(
    encode_batch,
    batched=True,
    remove_columns=["text"]
)

In [ ]:
print(encoded_dataset["train"][88]["ids"])
print([tokenizer.id_to_token(i) for i in encoded_dataset["train"][88]["ids"]])

[36, 3499, 3, 6, 18, 33, 69, 641, 20, 529, 3, 12, 1950, 2041, 19, 181, 27, 2, 3335, 444, 27, 2, 393, 461, 2672, 19, 18, 2313, 4, 16, 14738, 16, 2, 0, 0, 5, 993, 7, 2, 404, 3, 2, 989, 80, 34, 433, 18, 25, 12, 149, 2024, 8, 181, 295, 3600, 5, 2, 123, 2470, 19, 13353, 1513, 4, 26435, 30, 2052, 50, 25482, 3, 5242, 2202, 50, 3787, 3, 6, 2, 4915, 5, 2, 845, 557, 69, 641, 20, 529, 6, 78, 31, 4019, 122, 149, 27, 393, 461, 5164, 27, 396, 4, 39, 3, 48, 19685, 2, 143, 6, 6934, 2, 2658, 5, 2, 261, 69, 609, 18, 9, 3850, 720, 547, 152, 1328, 87, 125, 17416, 6, 18, 69, 67, 421, 2, 2591, 489, 78, 8817, 489, 8, 15912, 13, 736, 23568, 16, 626, 16, 23698, 6, 2694, 152, 34, 3106, 26, 393, 461, 442, 5, 0, 6, 22648, 3, 20, 33, 74, 5, 99, 30, 1397, 4, 51, 341, 0, 95, 2, 861, 152, 34, 17324, 4, 11, 1]
['this', 'ammunition', ',', 'and', 'that', 'which', 'i', 'brought', 'with', 'me', ',', 'was', 'rapidly', 'prepared', 'for', 'use', 'at', 'the', 'laboratory', 'established', 'at', 'the', 'little', 'rock', 'arsena

## Organización de las secuencias

`get_data` concatena los textos codificados de una partición y organiza los índices en un tensor `(batch_size, T)`. Cada fila es una secuencia continua que se recorrerá por bloques durante el entrenamiento.

Descartamos los tokens sobrantes que impiden dividir el total entre `batch_size`. No añadimos relleno ni conservamos cada noticia o artículo como un ejemplo independiente.

In [ ]:
def get_data(split, batch_size):
    data = []
    for example in split:
        data.extend(example["ids"])

    data = torch.tensor(data, dtype=torch.long)
    num_batches = data.shape[0] // batch_size
    data = data[:num_batches * batch_size]
    data = data.view(batch_size, num_batches)
    return data

## Generación de texto

`generate` codifica y procesa la frase inicial una sola vez. Después introduce únicamente el token recién generado, conservando los estados de la LSTM para mantener el contexto.

Dividimos los logits entre una temperatura positiva y aplicamos `softmax`: valores bajos concentran las probabilidades y valores mayores permiten más variedad. Excluimos `<unk>` antes de muestrear y terminamos al obtener `<eos>` o alcanzar `max_seq_len` tokens nuevos.

La función trabaja en modo de evaluación, sin gradientes, y admite una semilla opcional. Rechaza frases sin tokens y devuelve los tokens iniciales junto con los generados.

In [ ]:
def generate(prompt, max_seq_len, temperature, model, tokenizer, device, seed=None):
    if temperature <= 0:
        raise ValueError("La temperatura debe ser mayor que cero.")

    if seed is not None:
        torch.manual_seed(seed)

    model.eval()

    indices = tokenizer.encode(prompt).ids
    if not indices:
        raise ValueError("La frase inicial debe contener al menos un token.")

    unk_id = tokenizer.token_to_id("<unk>")
    eos_id = tokenizer.token_to_id("<eos>")
    hidden = model.init_hidden(1, device)

    # Primero procesamos toda la frase inicial.
    src = torch.tensor([indices], dtype=torch.long, device=device)

    with torch.no_grad():
        for _ in range(max_seq_len):
            prediction, hidden = model(src, hidden)
            logits = prediction[:, -1, :] / temperature

            # Excluimos <unk> antes de calcular las probabilidades.
            if unk_id is not None:
                logits[:, unk_id] = -float("inf")

            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).item()

            if next_id == eos_id:
                break

            indices.append(next_id)

            # El estado ya contiene el contexto anterior.
            src = torch.tensor(
                [[next_id]], dtype=torch.long, device=device
            )

    return [tokenizer.id_to_token(i) for i in indices]

## Preparación de las particiones

Organizamos entrenamiento, validación y prueba en 128 secuencias paralelas por partición. Los tensores permanecen en CPU; cada bloque se trasladará al dispositivo al procesarlo.

In [ ]:
batch_size = 128

train_data = get_data(encoded_dataset['train'], batch_size)
valid_data = get_data(encoded_dataset['validation'], batch_size)
test_data = get_data(encoded_dataset['test'], batch_size)

## Arquitectura del modelo

La red transforma índices `(B, L)` en embeddings, procesa la secuencia con una LSTM y produce scores sin normalizar (*logits*) de forma `(B, L, vocab_size)`. Cada posición predice el token siguiente.

Aplicamos *dropout* a los embeddings, entre capas LSTM y a la salida recurrente. Con `tie_weights=True`, el embedding y la capa de salida comparten pesos; sus dimensiones deben coincidir.

`init_hidden` crea los estados oculto y de memoria en ceros. `detach_hidden` conserva sus valores y desconecta el historial de gradientes entre bloques.

In [ ]:
class LSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout_rate,
                tie_weights):

        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.embedding_dim = embedding_dim

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers,
                    dropout=dropout_rate, batch_first=True)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_dim, vocab_size)

        if tie_weights:
            assert embedding_dim == hidden_dim, 'cannot tie, check dims'
            self.embedding.weight = self.fc.weight
        self.init_weights()

    def forward(self, src, hidden):
        embedding = self.dropout(self.embedding(src))
        output, hidden = self.lstm(embedding, hidden)
        output = self.dropout(output)
        prediction = self.fc(output)
        return prediction, hidden

    def init_weights(self):
        init_range_emb = 0.1
        init_range_other = 1/math.sqrt(self.hidden_dim)
        self.embedding.weight.data.uniform_(-init_range_emb, init_range_emb)
        self.fc.weight.data.uniform_(-init_range_other, init_range_other)
        self.fc.bias.data.zero_()
        for i in range(self.num_layers):
            self.lstm.all_weights[i][0] = torch.FloatTensor(self.embedding_dim,
                    self.hidden_dim).uniform_(-init_range_other, init_range_other)
            self.lstm.all_weights[i][1] = torch.FloatTensor(self.hidden_dim,
                    self.hidden_dim).uniform_(-init_range_other, init_range_other)

    def init_hidden(self, batch_size, device):
        hidden = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)
        cell = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)
        return hidden, cell

    def detach_hidden(self, hidden):
        hidden, cell = hidden
        hidden = hidden.detach()
        cell = cell.detach()
        return hidden, cell

## Hiperparámetros, dispositivo e inicialización

Usamos embeddings y estados ocultos de 1024 componentes, dos capas LSTM, dropout de `0.65` y pesos compartidos. Adam comienza con una tasa de aprendizaje de `0.001`; usamos cross-entropy como función de costo.

In [ ]:
embedding_dim = 1024             # 400 in the paper
hidden_dim = 1024                # 1150 in the paper
num_layers = 2                   # 3 in the paper
dropout_rate = 0.65
tie_weights = True
lr = 1e-3                        # They used 30 and a different optimizer

In [ ]:
import platform
if platform.system() == 'Darwin': #si es macOS
    av1 = torch.backends.mps.is_available()
    # this ensures that the current PyTorch installation was built with MPS activated.
    av2 = torch.backends.mps.is_built()
    if(av1==av2==True):
        print('MPS available?',av1,'MPS built?',av2)
        device = torch.device('mps')
    else:
        device = torch.device('cpu')
else:
    #Check if GPU is available
    if torch.cuda.is_available():
        device = torch.device('cuda')
    else:
        device = torch.device('cpu')
print("Device:",device)


MPS available? True MPS built? True
Device: mps


In [ ]:
model = LSTM(vocab_size, embedding_dim, hidden_dim, num_layers, dropout_rate, tie_weights).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'The model has {num_params:,} trainable parameters')

The model has 47,006,500 trainable parameters


## Inputs y outputs

`get_batch` extrae un bloque de `seq_len` tokens y otro desplazado una posición. Por ejemplo, para la entrada «el gato come», el objetivo sería «gato come pescado»: cada posición aprende a predecir el token siguiente.

In [ ]:
def get_batch(data, seq_len, num_batches, idx):
    src = data[:, idx:idx+seq_len]
    target = data[:, idx+1:idx+seq_len+1]
    return src, target

## Batch training

Activamos el modo de entrenamiento y recorremos bloques de texto consecutivos, conservando los estados de la LSTM entre ellos. Desconectamos los gradientes al comenzar cada bloque para limitar backprop a ese fragmento.

In [ ]:
def train(model, data, optimizer, criterion, batch_size, seq_len, clip, device):

    epoch_loss = 0
    model.train()
    # drop all batches that are not a multiple of seq_len
    num_batches = data.shape[-1]
    data = data[:, :num_batches - (num_batches -1) % seq_len]
    num_batches = data.shape[-1]

    hidden = model.init_hidden(batch_size, device)

    for idx in tqdm(range(0, num_batches - 1, seq_len), desc='Training: ',leave=False):  # The last batch can't be a src
        optimizer.zero_grad()
        hidden = model.detach_hidden(hidden)

        src, target = get_batch(data, seq_len, num_batches, idx)
        src, target = src.to(device), target.to(device)
        batch_size = src.shape[0]
        prediction, hidden = model(src, hidden)

        prediction = prediction.reshape(batch_size * seq_len, -1)
        target = target.reshape(-1)
        loss = criterion(prediction, target)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item() * seq_len
    return epoch_loss / num_batches

## Evaluación

`evaluate` recorre los bloques sin calcular gradientes y con el dropout desactivado. Inicia los estados en ceros y los conserva entre bloques, sin actualizar los pesos.

Se calcula la pérdida, dividida entre la longitud de la secuencia recortada. Su exponencial se usa como perplejidad: valores menores indican mejores predicciones del siguiente token. Esta normalización incluye una posición que no tiene objetivo, por lo que es una aproximación a la pérdida media por token predicho.

In [ ]:
def evaluate(model, data, criterion, batch_size, seq_len, device):

    epoch_loss = 0
    model.eval()
    num_batches = data.shape[-1]
    data = data[:, :num_batches - (num_batches -1) % seq_len]
    num_batches = data.shape[-1]

    hidden = model.init_hidden(batch_size, device)

    with torch.no_grad():
        for idx in range(0, num_batches - 1, seq_len):
            hidden = model.detach_hidden(hidden)
            src, target = get_batch(data, seq_len, num_batches, idx)
            src, target = src.to(device), target.to(device)
            batch_size= src.shape[0]

            prediction, hidden = model(src, hidden)
            prediction = prediction.reshape(batch_size * seq_len, -1)
            target = target.reshape(-1)

            loss = criterion(prediction, target)
            epoch_loss += loss.item() * seq_len
    return epoch_loss / num_batches

## Entrenamiento o carga de pesos

Si existe `best-val-lstm_lm.pt` en el directorio de trabajo, cargamos sus pesos y mostramos la perplejidad de prueba. Debe corresponder a la misma arquitectura y al mismo vocabulario.

Si no existe, entrenamos durante 30 épocas con bloques de 50 tokens y límite de norma de gradiente `0.25`. Reducimos a la mitad la tasa de aprendizaje cuando la pérdida de validación deja de mejorar y guardamos los pesos con menor pérdida de validación.

Tras entrenar desde cero, el modelo en memoria conserva los pesos de la última época: esta rama no recarga los mejores pesos ni evalúa el conjunto de prueba.

In [ ]:
from pathlib import Path

n_epochs = 30
seq_len = 50
clip = 0.25
#saved = True
model_path = Path("best-val-lstm_lm.pt")
saved = model_path.exists()


lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=0)

if saved:
    model.load_state_dict(torch.load('best-val-lstm_lm.pt',  map_location=device))
    test_loss = evaluate(model, test_data, criterion, batch_size, seq_len, device)
    print(f'Test Perplexity: {math.exp(test_loss):.3f}')
else:
    print("No hay modelo guardado; entrenando desde cero.")
    best_valid_loss = float('inf')

    for epoch in range(n_epochs):
        train_loss = train(model, train_data, optimizer, criterion,
                    batch_size, seq_len, clip, device)
        valid_loss = evaluate(model, valid_data, criterion, batch_size,
                    seq_len, device)

        lr_scheduler.step(valid_loss)

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            torch.save(model.state_dict(), './best-val-lstm_lm.pt')

        print(f'\tTrain Perplexity: {math.exp(train_loss):.3f}')
        print(f'\tValid Perplexity: {math.exp(valid_loss):.3f}')

Test Perplexity: 140.445


## Comparación de temperaturas

Partimos de `Think about` y solicitamos hasta 30 tokens nuevos con temperaturas de `0.5`, `0.7`, `0.75`, `0.8` y `1.0`. Fijamos la semilla en 0 en cada llamada para facilitar la comparación del muestreo.

Usamos `generate` para observar cómo cambia el texto al modificar la concentración de las probabilidades.

In [ ]:
prompt = 'Think about'
max_seq_len = 30
seed = 0

temperatures = [0.5, 0.7, 0.75, 0.8, 1.0]
for temperature in temperatures:
    generation = generate(prompt, max_seq_len, temperature, model, tokenizer,
                      device, seed)
    print(str(temperature)+'\n'+' '.join(generation)+'\n')

0.5
think about the same time , and the most common life , who had been able to be a child . "

0.7
think about the same time , and the most common life , who had been able to get rid of the blood and the earth and the curtain . it is sometimes

0.75
think about the same time , and only the claims of the group of the sun ' s second and the blood is a small and realistic cliché . it is sometimes

0.8
think about the same generation , and only the claims of the group of the sun ' s second and the blood is a small and realistic cliché . it is sometimes

1.0
think about something of human notes , because the claims wasn ' t be very able to hollywood after not know pleading for him to remove her readers . it had been

